# ESL Error Detector — Colab Training
Run each cell in order. Make sure your runtime is set to a GPU (Runtime → Change runtime type → T4 GPU).

In [11]:
# check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [12]:
!pip install -q transformers datasets scikit-learn pandas

In [13]:
# mount drive to save checkpoint
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/esl_checkpoint'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# load full dataset (no cap)
from datasets import load_dataset
import pandas as pd

ds = load_dataset('512duncanl/wi_locness_detokenized', split='train')
df = pd.DataFrame({'original_text': ds['input'], 'corrected_text': ds['output']})
print(f'Loaded {len(df)} examples')

Loaded 38692 examples


In [18]:
# filter near-identical pairs and build labeled dataset
from difflib import SequenceMatcher

def diff_ratio(a, b):
    return 1 - SequenceMatcher(None, a, b).ratio()

df['diff_ratio'] = df.apply(lambda x: diff_ratio(str(x['original_text']), str(x['corrected_text'])), axis=1)

before = len(df)
df = df[df['diff_ratio'] >= 0.02].reset_index(drop=True)
print(f'Filtered {before - len(df)} near-identical pairs, {len(df)} remaining')

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
half = len(df) // 2

err_examples = pd.DataFrame({'text': df.iloc[:half]['original_text'], 'label': 1})
ok_examples  = pd.DataFrame({'text': df.iloc[half:]['corrected_text'], 'label': 0})

combined = pd.concat([err_examples, ok_examples], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset size: {len(combined)}')
print(combined['label'].value_counts())

Filtered 0 near-identical pairs, 12667 remaining
Dataset size: 12667
label
0    6334
1    6333
Name: count, dtype: int64


In [19]:
# tokenize
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ds = Dataset.from_pandas(combined[['text', 'label']])
split = ds.train_test_split(test_size=0.2)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized = split.map(tokenize, batched=True)
print('Tokenization done')

Map:   0%|          | 0/10133 [00:00<?, ? examples/s]

Map:   0%|          | 0/2534 [00:00<?, ? examples/s]

Tokenization done


In [20]:
# train with class weights to push recall up
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, precision_score, recall_score
import torch.nn as nn
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# weight ERR class higher so the model is penalized more for missing errors
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        weights = torch.tensor([1.0, 1.5]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    return {
        'accuracy':  float((preds == labels).mean()),
        'f1':        f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall':    recall_score(labels, preds),
    }

args = TrainingArguments(
    output_dir=SAVE_PATH,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.515113,0.585964,0.744278,0.705722,0.865256,0.595859
2,0.321099,0.503882,0.764009,0.768934,0.774922,0.763037
3,0.143266,0.836828,0.765588,0.756158,0.813604,0.706288
4,0.064569,1.343940,0.747040,0.714477,0.852285,0.615031
5,0.033804,1.331148,0.758090,0.744902,0.814377,0.686350


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1585, training_loss=0.22750757718311876, metrics={'train_runtime': 648.695, 'train_samples_per_second': 78.103, 'train_steps_per_second': 2.443, 'total_flos': 1677865188226560.0, 'train_loss': 0.22750757718311876, 'epoch': 5.0})

In [21]:
# evaluate and print final metrics
results = trainer.evaluate()
print('\n=== Final Eval Results ===')
for k, v in results.items():
    print(f'{k}: {v:.4f}')


=== Final Eval Results ===
eval_loss: 0.5040
eval_accuracy: 0.7648
eval_f1: 0.7695
eval_precision: 0.7761
eval_recall: 0.7630
eval_runtime: 8.9083
eval_samples_per_second: 284.4530
eval_steps_per_second: 8.9800
epoch: 5.0000


In [22]:
# find best classification threshold on eval set
import torch.nn.functional as F

pred_output = trainer.predict(tokenized['test'])
probs = F.softmax(torch.tensor(pred_output.predictions), dim=-1)[:, 1].numpy()
true_labels = pred_output.label_ids

print(f"{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 55)
best_f1, THRESHOLD = 0, 0.5
for thresh in [0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (probs >= thresh).astype(int)
    acc  = float((preds == true_labels).mean())
    f1   = f1_score(true_labels, preds)
    prec = precision_score(true_labels, preds)
    rec  = recall_score(true_labels, preds)
    print(f"{thresh:>10.2f} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")
    if f1 > best_f1:
        best_f1, THRESHOLD = f1, thresh

print(f'\nBest threshold: {THRESHOLD} (F1: {best_f1:.4f})')

 Threshold   Accuracy  Precision     Recall         F1
-------------------------------------------------------
      0.30     0.7541     0.7201     0.8543     0.7815
      0.35     0.7612     0.7373     0.8328     0.7821
      0.40     0.7652     0.7530     0.8090     0.7800
      0.45     0.7636     0.7617     0.7868     0.7740
      0.50     0.7648     0.7761     0.7630     0.7695

Best threshold: 0.35 (F1: 0.7821)


In [23]:
# save model and tokenizer to drive
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved to {SAVE_PATH}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/esl_checkpoint


In [24]:
# quick sanity check using tuned threshold
LABEL_LIST = ['OK', 'ERR']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

test_sentences = [
    ('She go to school yesterday.', 'ERR'),
    ('He enjoys playing basketball.', 'OK'),
    ('I has a apple.', 'ERR'),
    ('The cat sat on the mat.', 'OK'),
    ('Me and him went to the park.', 'ERR'),
    ('We are watching a movie tonight.', 'OK'),
    ('They was late to the meeting.', 'ERR'),
    ('This is a beautiful day.', 'OK'),
    ("Him don't like vegetables.", 'ERR'),
    ('She is going to the store.', 'OK'),
]

correct = 0
for text, expected in test_sentences:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    prob_err = F.softmax(logits, dim=-1)[0, 1].item()
    pred = 'ERR' if prob_err >= THRESHOLD else 'OK'
    status = 'V' if pred == expected else 'X'
    print(f'{status} [{expected} -> {pred}] (p={prob_err:.2f}) {text}')
    if pred == expected:
        correct += 1

print(f'\nAccuracy: {correct}/{len(test_sentences)} (threshold={THRESHOLD})')

V [ERR -> ERR] (p=0.99) She go to school yesterday.
V [OK -> OK] (p=0.27) He enjoys playing basketball.
V [ERR -> ERR] (p=0.53) I has a apple.
X [OK -> ERR] (p=0.37) The cat sat on the mat.
V [ERR -> ERR] (p=0.75) Me and him went to the park.
V [OK -> OK] (p=0.20) We are watching a movie tonight.
X [ERR -> OK] (p=0.34) They was late to the meeting.
V [OK -> OK] (p=0.18) This is a beautiful day.
V [ERR -> ERR] (p=0.99) Him don't like vegetables.
X [OK -> ERR] (p=0.37) She is going to the store.

Accuracy: 7/10 (threshold=0.35)
